# Ashaar dataset exploration

In [1]:
from pathlib import Path

import pandas as pd

dataset_path = Path("../data/ashaar_dataset.parquet")
ashaar = pd.read_parquet(dataset_path)

## First 5 samples

In [2]:
ashaar.head(5)

,poem_title,poem_meter,poem_verses,poem_theme,poem_url,poet_name,poet_description,poet_url,poet_era,poet_location,poem_description,poem_language_type,text,split
0,أصبح الملك للذي فطر الخلق,1,"[أَصبَحَ المُلك لِلَّذي فَطر الخَل, قَ بِتَقدي...",قصيدة دينية,https://www.aldiwan.net/poem16182.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,NaN,None,NaN,<|meter_0|> م <|theme_18|> <|psep|> <|bsep|> أ...,train
1,من أي مولى ارتجي,3,"[مِن أَي مَولى اِرتَجي, وَلاي باب التَجي, وَال...",قصيدة دينية,https://www.aldiwan.net/poem16183.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,NaN,None,NaN,<|meter_3|> ج <|theme_18|> <|psep|> <|bsep|> م...,train
2,لو كنت أطمع بالمنام توهما,6,"[لَو كُنتَ أَطمَع بِالمَنام تَوهما, لَسالَت طَ...",قصيدة عامه,https://www.aldiwan.net/poem16185.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,NaN,None,NaN,<|meter_14|> م <|theme_17|> <|psep|> <|bsep|> ...,train
3,يعد علي أنفاسي ذنوبا,16,"[يعد عَليَّ أَنفاسي ذُنوباً, ِذا ما قُلت أَفدي...",قصيدة عامه,https://www.aldiwan.net/poem16186.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,NaN,None,NaN,<|meter_6|> ب <|theme_17|> <|psep|> <|bsep|> ي...,train
4,واها لموقفنا ببرقة تهمد,6,"[واهاً لِموقفنا بِبرقة تَهمدِ, بَينَ النَواهد ...",قصيدة عامه,https://www.aldiwan.net/poem16187.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,NaN,None,NaN,<|meter_14|> د <|theme_17|> <|psep|> <|bsep|> ...,train


## Dataset info

In [3]:
ashaar.info()

<class 'pandas.DataFrame'>
RangeIndex: 212499 entries, 0 to 212498
Data columns (total 14 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   poem_title          125621 non-null  str   
 1   poem_meter          212499 non-null  int64 
 2   poem_verses         212499 non-null  object
 3   poem_theme          60891 non-null   str   
 4   poem_url            212461 non-null  str   
 5   poet_name           212499 non-null  str   
 6   poet_description    62912 non-null   str   
 7   poet_url            150728 non-null  str   
 8   poet_era            136700 non-null  str   
 9   poet_location       56922 non-null   str   
 10  poem_description    13717 non-null   object
 11  poem_language_type  148445 non-null  str   
 12  text                212499 non-null  str   
 13  split               212499 non-null  str   
dtypes: int64(1), object(2), str(11)
memory usage: 487.4+ MB


## Classical Arabic subset

For this notebook, *classical* means that the poet has a known pre-modern era and the poem is explicitly labelled `فصيح` or `فصحى`. Rows with missing era or language metadata are intentionally excluded.

In [4]:
MODERN_ERA = "العصر الحديث"
FORMAL_ARABIC_LABELS = {"فصيح", "فصحى"}

classic_mask = (
    ashaar["poet_era"].notna()
    & ashaar["poet_era"].ne(MODERN_ERA)
    & ashaar["poem_language_type"].isin(FORMAL_ARABIC_LABELS)
)
classic_ashaar = ashaar.loc[classic_mask].copy().reset_index(drop=True)

assert len(classic_ashaar) == 49_004
assert classic_ashaar["poet_era"].notna().all()
assert classic_ashaar["poet_era"].ne(MODERN_ERA).all()
assert classic_ashaar["poem_language_type"].isin(FORMAL_ARABIC_LABELS).all()
assert list(classic_ashaar.columns) == list(ashaar.columns)

In [5]:
classic_summary = pd.DataFrame(
    {
        "value": [
            len(classic_ashaar),
            classic_ashaar["poet_name"].nunique(),
            classic_ashaar["poet_era"].nunique(),
        ]
    },
    index=["poems", "unique_poets", "eras"],
)
display(classic_summary)
display(classic_ashaar["poet_era"].value_counts().rename("poems").to_frame())
display(
    classic_ashaar[
        ["poem_title", "poet_name", "poet_era", "poem_language_type"]
    ].head(5)
)

,value
poems,49004
unique_poets,3466
eras,11


,poems
poet_era,
العصر العباسي,12059
العصر المملوكي,7367
المغرب والأندلس,6461
العصر العثماني,6193
العصر الفاطمي,4865
العصر الأموي,3828
العصر الأيوبي,2986
المخضرمين,1881
قبل الإسلام,1672


,poem_title,poet_name,poet_era,poem_language_type
0,NaN,عامر العَدواني,قبل الإسلام,فصيح
1,NaN,عامر العَدواني,قبل الإسلام,فصيح
2,NaN,عامر العَدواني,قبل الإسلام,فصيح
3,NaN,عامر العَدواني,قبل الإسلام,فصيح
4,NaN,عامر العَدواني,قبل الإسلام,فصيح


## Moroccan-origin authors

An author is included immediately when any of their rows has `poet_location == "المغرب"`; all poems by that author are then retained. Biography keywords only create a review queue and never include an author automatically. The review below uses present-day Morocco: nationality, birth, or ancestry tied to Morocco qualifies, while travel, residence, death location, manuscript location, generic historical Maghreb references, and ambiguous cases do not.

In [6]:
import re

MOROCCO = "المغرب"
MOROCCAN_BIOGRAPHY_TERMS = [
    MOROCCO,
    "مغربي",
    "مغربية",
    "المغاربة",
    "الدار البيضاء",
    "فاس",
    "مراكش",
    "الرباط",
    "تطوان",
    "مكناس",
    "مكناسة",
    "سلا",
    "طنجة",
    "وجدة",
    "أكادير",
    "الصويرة",
    "آسفي",
    "تارودانت",
    "شفشاون",
    "تازة",
    "أزمور",
    "أغمات",
    "دكالة",
    "تزنيت",
    "الصمارة",
    "القنيطرة",
    "الجديدة",
    "الحسيمة",
    "الناظور",
    "بني ملال",
    "ورزازات",
    "خنيفرة",
    "تافيلالت",
    "تادلة",
    "درعة",
    "سوس",
    "مولاي إدريس",
    "خريبكة",
    "العرائش",
    "أصيلة",
    "إفران",
    "إفني",
    "الريف",
]
moroccan_biography_pattern = re.compile(
    r"(?<![ء-ي])(?:"
    + "|".join(map(re.escape, MOROCCAN_BIOGRAPHY_TERMS))
    + r")(?![ء-ي])"
)

exact_location_poets = set(
    ashaar.loc[ashaar["poet_location"].eq(MOROCCO), "poet_name"]
)
author_biographies = (
    ashaar.loc[
        ashaar["poet_description"].notna()
        & ~ashaar["poet_name"].isin(exact_location_poets),
        ["poet_name", "poet_location", "poet_description"],
    ]
    .drop_duplicates("poet_name")
    .copy()
)
biography_candidate_mask = author_biographies["poet_description"].map(
    lambda description: bool(moroccan_biography_pattern.search(description))
)
moroccan_biography_candidates = author_biographies.loc[
    biography_candidate_mask
].copy()
moroccan_biography_candidates["trigger_terms"] = moroccan_biography_candidates[
    "poet_description"
].map(
    lambda description: sorted(
        set(moroccan_biography_pattern.findall(description))
    )
)
moroccan_biography_candidates = moroccan_biography_candidates.sort_values(
    "poet_name"
).reset_index(drop=True)

assert len(exact_location_poets) == 53
assert len(moroccan_biography_candidates) == 37
display(
    moroccan_biography_candidates[
        ["poet_name", "poet_location", "trigger_terms", "poet_description"]
    ]
)

,poet_name,poet_location,trigger_terms,poet_description
0,أبو العباس الجراوي,NaN,"[تادلة, مراكش]",أحمد بن عبد السلام الجراوي، أبو العباس.\nشاعر،...
1,أبو العباسِ الجَراوي,NaN,"[تادلة, مراكش]",نبذة\n\t\t\t:\n\t\t\tأحمد بن عبد السلام الجراو...
2,أبو العلاء المعري,NaN,[الرباط],أحمد بن عبد الله بن سليمان، التنوخي المعري.\nش...
3,أبو بكر بن مجبر,NaN,"[المغرب, مراكش]",عبد الجليل بن عبد الرحمن بن مجير الفهري، أبو ب...
4,أَحمَد بن المَأمون البلغيثي,NaN,"[الدار البيضاء, الصويرة, فاس, مكناسة]",نبذة\n\t\t\t:\n\t\t\tأحمد بن المأمون البلغيثي ...
5,إبراهيم اليازجي,لبنان,[المغرب],إبراهيم بن ناصيف بن عبد الله بن ناصيف بن جنبلا...
6,ابن الأبار البلنسي,NaN,[المغرب],محمد بن عبد الله بن أبي بكر القضاعي البلنسي أب...
7,ابن الأثير المحدث,NaN,[الرباط],المبارك بن محمد بن محمد بن محمد ابن عبد الكريم...
8,ابن الحاج النميري,NaN,[المغرب],نبذة\n\t\t\t:\n\t\t\tإبراهيم بن عبد الله بن إب...
9,ابن الحاجب النحوي,NaN,[الرباط],عثمان بن عمر بن أبي بكر بن يونس، أبو عمرو جمال...


### Reviewed biography candidates

The 37 author-level candidates were checked against the linked external biographies on 2026-08-03. Eight people are approved under nine exact dataset spellings; the remaining candidates are rejected. Ambiguous broad-Maghreb ancestry is rejected unless a source connects it to present-day Morocco.

In [7]:
moroccan_author_review = pd.DataFrame(
    [
        {"poet_name": "أبو العباس الجراوي", "is_moroccan": True, "rationale": "Born in Tadla in present-day Morocco.", "source_url": "https://www.habous.gov.ma/daouat-alhaq/item/3105"},
        {"poet_name": "أبو العباسِ الجَراوي", "is_moroccan": True, "rationale": "Diacritized dataset variant of Abu al-Abbas al-Jarawi, born in Tadla.", "source_url": "https://www.habous.gov.ma/daouat-alhaq/item/3105"},
        {"poet_name": "أبو العلاء المعري", "is_moroccan": False, "rationale": "Born and died in Maarrat al-Numan; Rabat is only a manuscript location.", "source_url": "https://poetry.dct.gov.ae/poets/412-%D8%A3%D8%A8%D9%88-%D8%A7%D9%84%D8%B9%D9%84%D8%A7%D8%A1-%D8%A7%D9%84%D9%85%D8%B9%D8%B1%D9%8A"},
        {"poet_name": "أبو بكر بن مجبر", "is_moroccan": False, "rationale": "Andalusian from Velez-Malaga; Marrakesh was a later residence.", "source_url": "https://www.aldiwan.net/cat-poet-abu-bakr-ibn-mogbar"},
        {"poet_name": "أَحمَد بن المَأمون البلغيثي", "is_moroccan": True, "rationale": "Born in Fez and served as a judge in Moroccan cities.", "source_url": "https://www.habous.gov.ma/daouat-alhaq/item/6314"},
        {"poet_name": "إبراهيم اليازجي", "is_moroccan": False, "rationale": "Born in Beirut to a family from Homs; Morocco refers to printing type.", "source_url": "https://poetry.dct.gov.ae/poets/1036-%D8%A5%D8%A8%D8%B1%D8%A7%D9%87%D9%8A%D9%85-%D8%A7%D9%84%D9%8A%D8%A7%D8%B2%D8%AC%D9%8A"},
        {"poet_name": "ابن الأبار البلنسي", "is_moroccan": False, "rationale": "Born in Valencia and identified as Andalusian.", "source_url": "https://www.aldiwan.net/cat-poet-ibn-alabar"},
        {"poet_name": "ابن الأثير المحدث", "is_moroccan": False, "rationale": "From Jazirat Ibn Umar; Rabat is only a manuscript location.", "source_url": "https://www.aldiwan.net/cat-poet-ibn-alothir-amahdt"},
        {"poet_name": "ابن الحاج النميري", "is_moroccan": False, "rationale": "Born in Granada; Morocco appears through royal service.", "source_url": "https://poetry.dctabudhabi.ae/diwan/poet/2560"},
        {"poet_name": "ابن الحاجب النحوي", "is_moroccan": False, "rationale": "Born in Esna, Egypt; Rabat is only a manuscript location.", "source_url": "https://www.aldiwan.net/cat-poet-abn-alhajeb-alnhoi"},
        {"poet_name": "ابن الونان", "is_moroccan": True, "rationale": "Born, raised, and died in Fez.", "source_url": "https://www.habous.gov.ma/daouat-alhaq/item/4616"},
        {"poet_name": "ابن الياسمين", "is_moroccan": True, "rationale": "Described as being from Marrakesh and dying there.", "source_url": "https://arab-ency.com.sy/details/160969"},
        {"poet_name": "ابن جرج الذهبي", "is_moroccan": False, "rationale": "Andalusian from Valencia and Almeria; Morocco refers to an army.", "source_url": "https://www.aldiwan.net/cat-poet-ibn-grj-alzhba"},
        {"poet_name": "ابن دريد الأزدي", "is_moroccan": False, "rationale": "Born in Basra; Rabat is only a manuscript location.", "source_url": "https://poetry.dctabudhabi.ae/diwan/poet/1634"},
        {"poet_name": "ابن زاكور", "is_moroccan": True, "rationale": "Born in Fez to a family established there.", "source_url": "https://www.mithaqarrabita.ma/%D8%A7%D8%A8%D9%86-%D8%B2%D8%A7%D9%83%D9%88%D8%B1-%D8%A7%D9%84%D9%81%D8%A7%D8%B3%D9%8A?edition=1"},
        {"poet_name": "ابن زمرك", "is_moroccan": False, "rationale": "Born in Granada; Morocco refers to where a manuscript was seen.", "source_url": "https://www.aldiwan.net/cat-poet-ibn-zamrak"},
        {"poet_name": "ابن زهر الحفيد", "is_moroccan": False, "rationale": "Born in Seville; Marrakesh was his place of death.", "source_url": "https://www.aldiwan.net/cat-ibn-zahr-alhfid"},
        {"poet_name": "ابن زيدون", "is_moroccan": False, "rationale": "Born in Cordoba; Morocco appears only in a literary epithet.", "source_url": "https://poetry.dctabudhabi.ae/diwan/poet/374"},
        {"poet_name": "ابن عمرو الأغماتي", "is_moroccan": True, "rationale": "Born in Aghmat and raised in Fez.", "source_url": "https://www.aldiwan.net/cat-poet-ibn-amr-alogmati"},
        {"poet_name": "ابن عياش التجيبي", "is_moroccan": False, "rationale": "Identified as Andalusian; Marrakesh was a later residence.", "source_url": "https://www.aldiwan.net/cat-poet-ibn-ayyash-altchibey"},
        {"poet_name": "ابن هانئ الأندلسي", "is_moroccan": False, "rationale": "Born near Seville and identified as Andalusian.", "source_url": "https://www.aldiwan.net/cat-poet-ibn-Hani-Andalusia"},
        {"poet_name": "الشريشي السلوي", "is_moroccan": True, "rationale": "Born in Sale and raised in Marrakesh.", "source_url": "https://poetry.dctabudhabi.ae/diwan/poet/2609"},
        {"poet_name": "المعتمد بن عباد", "is_moroccan": False, "rationale": "Andalusian ruler of Seville who was exiled to Aghmat.", "source_url": "https://poetry.dctabudhabi.ae/diwan/poet/1039"},
        {"poet_name": "حازم القرطاجني", "is_moroccan": False, "rationale": "Born in Cartagena; Marrakesh was a stop during migration.", "source_url": "https://poetry.dctabudhabi.ae/diwan/poet/1521"},
        {"poet_name": "حسن قويدر الخليلي", "is_moroccan": True, "rationale": "His documented ancestors came from Morocco before moving through Hebron to Cairo.", "source_url": "https://www.hindawi.org/books/18683951/4.13/"},
        {"poet_name": "سيديا بن المختار", "is_moroccan": False, "rationale": "A scholar of Shinqit in present-day Mauritania who visited Marrakesh.", "source_url": "https://poetry.dctabudhabi.ae/diwan/poet/2739"},
        {"poet_name": "ظافر الحداد", "is_moroccan": False, "rationale": "From Alexandria; Rabat is only a manuscript location.", "source_url": "https://poetry.dctabudhabi.ae/diwan/poet/1117"},
        {"poet_name": "عبد القادر الجزائري", "is_moroccan": False, "rationale": "Born in Algeria; Morocco appears in accounts of his conflict.", "source_url": "https://poetry.dctabudhabi.ae/diwan/poet/1661"},
        {"poet_name": "عبد المحسن الكاظمي", "is_moroccan": False, "rationale": "Born in Baghdad and raised in Kadhimiya; the trigger is not origin evidence.", "source_url": "https://poetry.dctabudhabi.ae/diwan/poet/1273"},
        {"poet_name": "عبد الناصر الجوهري", "is_moroccan": False, "rationale": "Egyptian poet; Morocco is only a publication reference.", "source_url": "https://www.aldiwan.net/cat-poet-Abdel-Nasser_Al-Johari"},
        {"poet_name": "عبدالله الشبراوي", "is_moroccan": False, "rationale": "Born and based in Cairo; Rabat is only a manuscript location.", "source_url": "https://www.aldiwan.net/cat-poet-Shabrawy"},
        {"poet_name": "لسان الدين بن الخطيب", "is_moroccan": False, "rationale": "Born in Granada; Fez was a later residence and place of death.", "source_url": "https://poetry.dctabudhabi.ae/diwan/poet/1197"},
        {"poet_name": "ماء العينين", "is_moroccan": False, "rationale": "Born in the Hawd region of present-day Mauritania; later lived in Morocco.", "source_url": "https://poetry.dctabudhabi.ae/diwan/poet/3010"},
        {"poet_name": "محمد وفا", "is_moroccan": False, "rationale": "Born in Alexandria; broad Maghrebi ancestry is ambiguous and one cited account points to Sfax.", "source_url": "https://app.alreq.com/ar/Authors/Author/3718811f-a559-4138-0505-08d7902f2e13"},
        {"poet_name": "محمود درويش", "is_moroccan": False, "rationale": "Palestinian poet; the matched word is not Moroccan origin evidence.", "source_url": "https://www.aldiwan.net/cat-poet-mahmoud-darwish"},
        {"poet_name": "يزيد بن معاوية", "is_moroccan": False, "rationale": "Umayyad ruler born in Syria; Morocco appears in a conquest account.", "source_url": "https://www.aldiwan.net/cat-poet-yazid-bin-muawiyah"},
        {"poet_name": "يوسف النبهاني", "is_moroccan": False, "rationale": "Born in Ijzim, Palestine; Rabat is only a manuscript location.", "source_url": "https://poetry.dctabudhabi.ae/diwan/poet/1532"},
    ]
).sort_values("poet_name").reset_index(drop=True)

candidate_names = set(moroccan_biography_candidates["poet_name"])
reviewed_names = set(moroccan_author_review["poet_name"])
assert len(moroccan_author_review) == 37
assert moroccan_author_review["poet_name"].is_unique
assert reviewed_names == candidate_names
assert moroccan_author_review["is_moroccan"].notna().all()
assert moroccan_author_review["rationale"].str.strip().ne("").all()
assert moroccan_author_review["source_url"].str.match(r"https?://").all()
display(moroccan_author_review)

,poet_name,is_moroccan,rationale,source_url
0,أبو العباس الجراوي,True,Born in Tadla in present-day Morocco.,https://www.habous.gov.ma/daouat-alhaq/item/3105
1,أبو العباسِ الجَراوي,True,Diacritized dataset variant of Abu al-Abbas al...,https://www.habous.gov.ma/daouat-alhaq/item/3105
2,أبو العلاء المعري,False,Born and died in Maarrat al-Numan; Rabat is on...,https://poetry.dct.gov.ae/poets/412-%D8%A3%D8%...
3,أبو بكر بن مجبر,False,Andalusian from Velez-Malaga; Marrakesh was a ...,https://www.aldiwan.net/cat-poet-abu-bakr-ibn-...
4,أَحمَد بن المَأمون البلغيثي,True,Born in Fez and served as a judge in Moroccan ...,https://www.habous.gov.ma/daouat-alhaq/item/6314
5,إبراهيم اليازجي,False,Born in Beirut to a family from Homs; Morocco ...,https://poetry.dct.gov.ae/poets/1036-%D8%A5%D8...
6,ابن الأبار البلنسي,False,Born in Valencia and identified as Andalusian.,https://www.aldiwan.net/cat-poet-ibn-alabar
7,ابن الأثير المحدث,False,From Jazirat Ibn Umar; Rabat is only a manuscr...,https://www.aldiwan.net/cat-poet-ibn-alothir-a...
8,ابن الحاج النميري,False,Born in Granada; Morocco appears through royal...,https://poetry.dctabudhabi.ae/diwan/poet/2560
9,ابن الحاجب النحوي,False,"Born in Esna, Egypt; Rabat is only a manuscrip...",https://www.aldiwan.net/cat-poet-abn-alhajeb-a...


In [8]:
approved_biography_poets = set(
    moroccan_author_review.loc[
        moroccan_author_review["is_moroccan"], "poet_name"
    ]
)
rejected_biography_poets = reviewed_names - approved_biography_poets
moroccan_poets = exact_location_poets | approved_biography_poets
moroccan_ashaar = (
    ashaar.loc[ashaar["poet_name"].isin(moroccan_poets)]
    .copy()
    .reset_index(drop=True)
)

moroccan_result_poets = set(moroccan_ashaar["poet_name"])
assert exact_location_poets <= moroccan_result_poets
assert approved_biography_poets <= moroccan_result_poets
assert not (rejected_biography_poets & moroccan_result_poets)
assert set(moroccan_ashaar["poet_name"]) <= moroccan_poets
assert list(moroccan_ashaar.columns) == list(ashaar.columns)
assert len(moroccan_ashaar) == 2_157
assert moroccan_ashaar["poet_name"].nunique() == 62

In [9]:
moroccan_summary = pd.DataFrame(
    {
        "value": [
            len(moroccan_ashaar),
            moroccan_ashaar["poet_name"].nunique(),
            len(exact_location_poets),
            len(approved_biography_poets),
            len(rejected_biography_poets),
        ]
    },
    index=[
        "poems",
        "unique_poet_names",
        "exact_location_poets",
        "approved_biography_names",
        "rejected_biography_names",
    ],
)
display(moroccan_summary)
display(
    moroccan_ashaar["poem_language_type"]
    .value_counts(dropna=False)
    .rename("poems")
    .to_frame()
)
display(
    moroccan_ashaar["poet_era"]
    .value_counts(dropna=False)
    .rename("poems")
    .to_frame()
)
display(
    moroccan_ashaar[
        ["poem_title", "poet_name", "poet_location", "poet_era"]
    ].head(5)
)

,value
poems,2157
unique_poet_names,62
exact_location_poets,53
approved_biography_names,9
rejected_biography_names,28


,poems
poem_language_type,
فصيح,1510
NaN,647


,poems
poet_era,
العصر العثماني,1079
العصر الحديث,520
NaN,301
المغرب والأندلس,224
العصر المملوكي,26
العصر الأيوبي,6
العصر الفاطمي,1


,poem_title,poet_name,poet_location,poet_era
0,وللكفات في المجهول وجه,ابن الياسمين,NaN,NaN
1,الحمد لله على ما الهما,ابن الياسمين,NaN,NaN
2,أيها الفاسي أتى ريحك,ابن الياسمين,NaN,NaN
3,جاء الربيع,ابن الياسمين,NaN,NaN
4,عجبت لمن يراك وبعد هذا,ابن الياسمين,NaN,NaN


## Save and verify the filtered datasets

In [10]:
classic_output_path = dataset_path.with_name("ashaar_classic.parquet")
moroccan_output_path = dataset_path.with_name("ashaar_moroccan.parquet")

classic_ashaar.to_parquet(classic_output_path, index=False)
moroccan_ashaar.to_parquet(moroccan_output_path, index=False)

saved_classic_ashaar = pd.read_parquet(classic_output_path)
saved_moroccan_ashaar = pd.read_parquet(moroccan_output_path)

assert len(saved_classic_ashaar) == len(classic_ashaar)
assert len(saved_moroccan_ashaar) == len(moroccan_ashaar)
assert list(saved_classic_ashaar.columns) == list(ashaar.columns)
assert list(saved_moroccan_ashaar.columns) == list(ashaar.columns)
pd.testing.assert_frame_equal(
    saved_classic_ashaar, classic_ashaar, check_dtype=False
)
pd.testing.assert_frame_equal(
    saved_moroccan_ashaar, moroccan_ashaar, check_dtype=False
)

pd.DataFrame(
    {
        "path": [classic_output_path, moroccan_output_path],
        "rows": [len(saved_classic_ashaar), len(saved_moroccan_ashaar)],
        "unique_poets": [
            saved_classic_ashaar["poet_name"].nunique(),
            saved_moroccan_ashaar["poet_name"].nunique(),
        ],
    },
    index=["classic", "moroccan"],
)

,path,rows,unique_poets
classic,..\data\ashaar_classic.parquet,49004,3466
moroccan,..\data\ashaar_moroccan.parquet,2157,62


## Merge the classical and Moroccan subsets

Use a union mask on the original dataset so poems present in both subsets are selected only once.

In [ ]:
moroccan_mask = ashaar["poet_name"].isin(moroccan_poets)
merged_mask = classic_mask | moroccan_mask
merged_ashaar = ashaar.loc[merged_mask].copy().reset_index(drop=True)

overlap_count = int((classic_mask & moroccan_mask).sum())
expected_merged_rows = (
    len(classic_ashaar) + len(moroccan_ashaar) - overlap_count
)

assert len(merged_ashaar) == expected_merged_rows == 50_199
assert list(merged_ashaar.columns) == list(ashaar.columns)
assert classic_mask.sum() <= len(merged_ashaar)
assert moroccan_mask.sum() <= len(merged_ashaar)

merged_output_path = dataset_path.with_name(
    "ashaar_classic_moroccan.parquet"
)
merged_ashaar.to_parquet(merged_output_path, index=False)
saved_merged_ashaar = pd.read_parquet(merged_output_path)

pd.testing.assert_frame_equal(
    saved_merged_ashaar, merged_ashaar, check_dtype=False
)

pd.DataFrame(
    {
        "value": [
            len(classic_ashaar),
            len(moroccan_ashaar),
            overlap_count,
            len(merged_ashaar),
            merged_ashaar["poet_name"].nunique(),
            merged_output_path,
        ]
    },
    index=[
        "classical_rows",
        "moroccan_rows",
        "overlap_removed",
        "merged_rows",
        "unique_poets",
        "path",
    ],
)